In [ ]:
import logging
import time
from corriscope.fpga_array import FPGAArray
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
# import scipy
# import labpy

# Must use %matplotlib widget with jupyter-lab. Needs 'pip install ipympl'
# jupyter-notebook needs %matplotlib notebook
%matplotlib widget
plt.rcParams['figure.figsize'] = (9.5, 6)
logging.getLogger().setLevel(logging.DEBUG) # ensure the notebook will show all log levels 

In [ ]:
ca = await FPGAArray('crs 37:1', stderr_log_level='warn', mode='corr8', sync_method='local', ignore_missing_boards=True)
await ca.ib.get_fpga_firmware_timestamp_async()

In [ ]:
import numpy

In [ ]:
i = ca.ib[0]
scaler = i.chan[0].SCALER
funcgen = i.chan[0].FUNCGEN
fft = i.chan[0].FFT
ucap = i.UCAP
ucorr = i.CORR
Fs = i._sampling_frequency
# await i.set_local_data_port_number_async(41234)  # use a separate UDP data destination port in case someone else is using it
r = await ca.get_data_receiver_async()
# cr = i.get_corr_receiver()
# cr.ignore_packet_size = True
gpio=i.GPIO


In [ ]:
gpio.CORR_RESET

In [ ]:
i.get_temperatures()

In [ ]:
#labpy.find_all_instruments()
#sg = labpy.instruments.rigol.RigolDSG800('TCPIP::10.10.10.152')
# sg.set_frequency(3.2e9/16384*100)
#sg.set_output_enable(False)

In [ ]:
i.SYSMON.read_drp(0)

In [ ]:
# First , check the signal amplitude on the ADCs

In [ ]:
funcgen.RESET_STATS=1
funcgen.RESET_STATS=0

In [ ]:
print(f'ADC overvoltage = {gpio.ADC_OV}, overrange={gpio.ADC_OR}, Overflow count={funcgen.ADC_OVERFLOW_CTR}')

In [ ]:
r.socket

In [ ]:
# i.set_channelizer(data_source='adc')
i.set_channelizer(data_source='sin', freq=1, channels=[0])
i.chan[0].FUNCGEN.RIGHT_SHIFT=0
i.chan[0].FUNCGEN.LEFT_SHIFT=0
# funcgen.SHIFT=0
# ucap.OUTPUT_SOURCE_SEL = 0
i.start_data_capture(0.05, source='funcgen', mode=0, channels=None)
# ucap.MODE = 0  # single-channel mode, 16 frames/channel
t,d,c = r.read_raw_frames(ncap=16,flush=True, verbose=1)

In [ ]:
d.shape

In [ ]:
# i.set_channelizer(data_source='sin', freq=1)
# i.start_data_capture(0.03, source='adc', mode=3, channels=[0], select=True)
# ucap.MODE = 3  # single-channel mode, 16 frames/channel
# t,(d, *_),c = r.read_raw_frames(format='16', verbose=1)
fig, ax = plt.subplots()
ax.set_title('Output of the ADC, terminated amplifier chain')
ax.set_xlabel('Sample #')
ax.set_ylabel('ADC output')
ax.plot(d[0].T)
d.shape

In [ ]:
f'{t[0]} (0x{t[0]:08x})'

In [ ]:
g=ca.ib.GPU


In [ ]:
i.GPU

In [ ]:
i.sync()

In [ ]:
fft.MEASURED_PIPELINE_DELAY

In [ ]:
fft.PIPELINE_DELAY

In [ ]:
fig, ax = plt.subplots()
ax.set_title('Output of the ADC, terminated amplifier chain')
ax.set_xlabel('Sample #')
ax.set_ylabel('ADC output')
# ax.plot(d)
# d.shape

In [ ]:
funcgen.CLIP_WIDTH

In [ ]:
scaler.CHAN_FIFO_OVERFLOW_RESET=1
scaler.CHAN_FIFO_OVERFLOW_RESET=0
gpio.CORR_RESET=0
i.sync()

In [ ]:
[chan.SCALER.CHAN_FIFO_OVERFLOW for chan in i.chan]

In [ ]:
i.sync()
# [scaler.MON_WORD for _ in range(100)]

In [ ]:
for ch in i.chan:
    w=ch.SCALER.MON_WORD
    print(f'CH{ch}: scaler_rst={(w>>12)&1}, out_rst={(w>>11)&1},send={(w>>15)&1}, almost_empty={(w>>10)&1}, wr_count={w&0x3FF}') 

In [ ]:
scaler.MON_CTRL=1
[scaler.MON_WORD for _ in range(10)]

In [ ]:
scaler.MON_CLK_CTR

In [ ]:
[scaler.MON_BIT_CTR for _ in range(10)]

In [ ]:
scaler.MON_RESET_STATS=1
scaler.MON_RESET_STATS=0

scaler.MON_PACKET_LENGTH

In [ ]:
i.FreqCtr.status()

In [ ]:
b=funcgen.get_buffer()
bb=np.frombuffer(b, dtype=">i2")
fig, ax = plt.subplots()
ax.plot(bb)

In [ ]:
def hist_bins(data):
    return np.arange(data.min(), data.max() + 1.5) - 0.5

def hist_discrete(ax, data):
    ax.hist(data, bins=hist_bins(data))

In [ ]:
fig, ax = plt.subplots()
ax.set_title('Histogram of ADC data')
ax.set_xlabel('LSB')
ax.set_ylabel('# of occurences')
dd= d >> 2 # divide by 4 because we want the upper 14 bits
dd >>= 1 # remove the lsb because odd and even bits have largely different weights
hist_discrete(ax,dd)

In [ ]:
print(f'min={d.min()}, max={d.max()}, mean={d.mean()},std={d.std()}')

In [ ]:
fig, ax = plt.subplots()
ax.set_label(f'ADC output spectrogram')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Frequency [Hz]')
pxx,f,t,cax = ax.specgram(d, Fs=Fs, NFFT=1<<10, noverlap=0, mode='magnitude');
fig.colorbar(cax, label='dB');

In [ ]:
ff = np.fft.rfft(d[:16384])
ff.shape

In [ ]:
F = 2048
sg.set_frequency(3.2e9/16384*F)

In [ ]:
i.REFCLK.local_sync()

In [ ]:
funcgen.RESET_STATS = 1
funcgen.RESET_STATS = 0
fft.RESET_STATS = 1
fft.RESET_STATS = 0
# funcgen.SHIFT = 0
fft.FFT_SHIFT=0b0
scaler.BYPASS = 1
scaler.DATA_TYPE = 0 # capture top 16 bits of FFT  (16+16-bit output)
ucap.OUTPUT_SOURCE_SEL = 0
fft.BYPASS=0
i.start_data_capture(0.03, source='scaler')
map = []
for F in range(13):
    F=2**F
# for F in range(8180, 8192):
    FF2 = (F>>11) | ((F & 0b11111111111) << 2) 
    i.set_channelizer(data_source='sin', freq=F, channels=[0])
    # sg.set_frequency(3.2e9/16384*F)
    time.sleep(.1)
    print(f'ADC overvoltage = {gpio.ADC_OV}, overrange={gpio.ADC_OR}, ADC Overflow count={funcgen.ADC_OVERFLOW_CTR}, FFT Overflow count = {fft.OVERFLOW_COUNT}')
    _,(df, *_),_ = r.read_raw_frames(format='16+16', ncap=2, split=1, verbose=False)
    FF = np.argmax(abs(df[-1]))
    map.append((F,FF, FF2))
    # print(f'{F} => {FF}')
    # print(f'{F:04d} ({F:013b}), {FF:04d}({FF:013b}), FF2:{FF2:013b}')
    print(f'{F:04d} ({F:013b}) => {FF:04d}({FF:013b})')
for F, FF, FF2 in map:
    # print(f'{F:013b} -> {FF:013b} ({FF2:013b} : {FF2==FF})')
    print(f'{F:013b} -> {FF:013b}')

In [ ]:
print(f'ADC overvoltage = {gpio.ADC_OV}, overrange={gpio.ADC_OR}, Overflow count={funcgen.ADC_OVERFLOW_CTR}')

In [ ]:
F='adc'
if F == 'adc':
    i.set_channelizer(data_source='adc', channels=[0])
    F=0
elif F:
    i.set_channelizer(data_source='cos', freq=F, channels=[0])
else:
    i.set_channelizer(data_source='a', a=512, channels=[0])
b=np.full(16384, 2, dtype=np.int16);
# b[1]=8191
# i.set_channelizer(data_source='buffer', buffer=b, channels=[0])
# funcgen.set_buffer(b)
# i.set_channelizer(data_source='a', a=1000, channels=[0])

In [ ]:
b[:10]

In [ ]:
# Capture raw fft before gain so we can check how many bits are exercised
# fft.FFT_SHIFT=0b1111_1100_0000_00
# fft.FFT_SHIFT=0b1111_1111_0000_00
# fft.FFT_SHIFT=(1<<10)-1
# fft.FFT_SHIFT=0b1111_1110_0000_00
# fft.FFT_SHIFT=0b0101_0101_0101_01
#fft.FFT_SHIFT=0b1111_1110_0000_00
fft.FFT_SHIFT=0b1111_1111_1111_11
# fft.FFT_SHIFT=0

In [ ]:
fft.OVERFLOW_RESET = 1
fft.OVERFLOW_RESET = 0

In [ ]:
fft_overflows = sum(fft.OVERFLOW_COUNT for _ in range(100))
print(f'FFT overfows = {fft_overflows}')
print(f'FFT shift=0b{fft.FFT_SHIFT:014b}')

In [ ]:
scaler.USE_FLOAT_GAINS


In [ ]:
# Capture post-gain, pre-clipping FFT signal
# b=np.full(16384, 1, dtype=np.int16);
# i.set_channelizer(data_source='ab', a=1, b=-1, channels=[0])
i.set_gains(2.2*2**30)
# i.set_gains((np.arange(8192.)*(1<<14), None))
i.start_data_capture(0.03, source='scaler')
ucap.OUTPUT_SOURCE_SEL = 0
fft.BYPASS = 0
scaler.BYPASS = 1
scaler.DATA_TYPE = 1 # 0: raw adc, 1: 16+16 bits post-gain
scaler.FOUR_BITS = 0
scaler.SHIFT_LEFT = 0
# _,(df, *_),_ = r.read_raw_frames(format='16+16', ncap=10, split=1, verbose=0)
_,(df, *_),_ = r.read_raw_frames(format='16+16', ncap=1, split=1, verbose=0)

df.shape
print(f'FFT shift=0b{fft.FFT_SHIFT:014b}')

In [ ]:
df[0, 2047:2200]

In [ ]:
bin(4096)

In [ ]:
print(f'Peak magnitude {np.max(abs(df[-1]))} @ bin {np.argmax(abs(df[-1]))}')

In [ ]:
#FF = np.argmax(abs(df[-1]))
#print(f'{F} => {FF}')
#print(f'{F:013b}, {FF:013b}')

In [ ]:
# scaler.get_gain_table()

In [ ]:
fig, ax = plt.subplots()
fig.suptitle(f'Output of the FFT (SCALER bypass={scaler.BYPASS} data_type={scaler.DATA_TYPE})\n')
ax.set_title(f'ADC OR={gpio.ADC_OR}, ADC_OV={gpio.ADC_OV}, FFT overflow={fft.OVERFLOW_COUNT}, FFT shift = {fft.FFT_SHIFT:014b}', fontsize=8)
ax.set_xlabel('Sample #')
ax.set_ylabel('FFT output [dB]')
# ax.plot(20*np.log10(abs(df[0])))
ax.plot(df[-1].real[frev2], alpha=0.4, label='Re')
ax.plot(df[-1].imag[frev2], alpha=0.4, label='Im')
ax.plot(np.mean((abs(df)), 0)[frev2], label='Magnitude')
ax.legend()

In [ ]:
df[0,:10]

In [ ]:
fig, ax = plt.subplots()
ax.set_title('Histogram of the post-gain FFT\n'
             f'FFT shift = {fft.FFT_SHIFT:014b}')
ax.set_xlabel('LSB')
ax.set_ylabel('# of occurences')
# hist_discrete(ax, df[0].real)
ax.hist((df[0,1:].real), bins=100, alpha = 0.9, color='b');
ax.hist((df[0,1:].imag), bins=100, alpha = 0.5, color='r');

In [ ]:
int(f'{100:013b}'[::-1],2)

In [ ]:
print(f'{129: 013b}, {384:013b}')

In [ ]:
ds=df[0].real; n=len(ds); nlsat=(ds==-32768).sum(); nhsat=(ds==32767).sum()
print(f'min={ds.min()}, max={ds.max()}, -nsat={nlsat} ({nlsat/n*100:.3f}%), +nsat={nhsat} ({nhsat/n*100:.3f}%)')

In [ ]:
i.GPIO.ADC_CAL_FREEZE = 0
print(f'cal frozen={i.GPIO.ADC_CAL_FROZEN}')
print(f'cal signal={i.GPIO.ADC_CAL_SIGNAL_DETECT}')


In [ ]:
i.GPIO.CORR_RESET=1
i.GPIO.CORR_RESET=0
ucorr.OVERRUN


In [ ]:
ff = (f_rev & 0b1111111111100) | ((f_rev + np.arange(8192)) & 0b11)

In [ ]:
ib=ca.ib[0]
gpio.ANT_RESET=0
gpio.CORR_RESET=0


In [ ]:
ib.CORR.OUT_FRAME_CTR

In [ ]:
ib.CT.CT1.OUT_FRAME_CTR

In [ ]:
i.FreqCtr.status()

In [ ]:
ib.CORR.IN_FRAME_CTR

In [ ]:
#Get corr output
i.set_channelizer(data_source='adc', fft_bypass=False, scaler_bypass=False); i.set_gains(2**33)
# i.set_channelizer(data_source='ab', a=1, b=0, fft_bypass=True, scaler_bypass=False); i.set_gains(2**25)

# i.set_channelizer(data_source='ab', a=20, b=0)
# i.set_channelizer(data_source='noise', fft_bypass=True, scaler_bypass=False); i.set_gains(1.5*2**12)

ucap.OUTPUT_SOURCE_SEL = 1
ucorr.NO_ACCUM = 0
ucorr.AUTOCORR_ONLY = 0
ucorr.INTEGRATION_PERIOD = 16384-1
scaler.FOUR_BITS = 1
scaler.SATURATE_ON_MINUS_7 = 1
# fft.FFT_SHIFT=(1<<10)-1
# fft.BYPASS=0
# scaler.BYPASS=0
scaler.DATA_TYPE=0
scaler.CAP_DATA_TYPE = 0
scaler.ROUNDING_MODE = 1
scaler.USE_OFFSET_BINARY = 0
print(f'FFT shift=0b{fft.FFT_SHIFT:014b}')
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush_timeout=0.0001, soft_integ_period=1, number_of_results=1, verbose=1)


In [ ]:
fig, ax = plt.subplots()
ax.set_title(f'CORR output, Chan 0 autocorr')
# ax.set_xlabel('Frequency [Hz]')
ax.set_xlabel('Freq (MHz)')
ax.set_ylabel('CORR output (dB)')
freqs = (np.arange(8192)/16384*i._sampling_frequency)
# mags = abs(dd0[0,:,0].T)[f_rev]
mags = abs(dd0[0,:,8].T)[ff]
# mags = abs(dd0[0,:,0].T)
# line,=ax.plot(freqs/1e6,mags, label='ADC corr ROUND');
# line,=ax.plot(freqs[4:], mags[4:], label='ADC corr ROUND');
line,=ax.plot(freqs[4:],mags[4:], label='ADC corr ROUND');
# line,=ax.plot(f_rev, mags, '.',label='ADC corr ROUND');
# line,=ax.plot(freqs[4:]/1e6,10*np.log10(abs(dd0[0,4:,0].T)), label='ADC corr ROUND');

# def animate(i):
#     line.set_ydata(10*np.log10(abs(dd0[i,4:,0].T)))  # update the data.
#     return line,
# animate(1)

# ani = animation.FuncAnimation(fig, animate, interval=200, blit=True, save_count=50,repeat=True)
# ani = animation.FuncAnimation(fig, animate, interval=30, frames=dd0.shape[0], repeat=True)


In [ ]:
f_rev[:16]

In [ ]:
1600/16384*16383

In [ ]:
10*np.log10(50)

In [ ]:
f_rev[0:4]

In [ ]:
ani.save(f'{i.part_number}_SN{i.serial}_noise1.gif')

In [ ]:
print(f'harmonics of {Fs/1e6} MHz: {", ".join(str(n*Fs/8/1e6) for n in np.arange(8))}')

In [ ]:
# real-time correlator output
fig, ax = plt.subplots()
ax.set_title(f'CORR output, Chan 0 autocorr')
# ax.set_xlabel('Frequency [Hz]')
ax.set_xlabel('Freq (MHz)')
ax.set_ylabel('CORR output')
freqs = np.arange(8192)/16384*i._sampling_frequency
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush_timeout=0.01, soft_integ_period=1, number_of_results=1, align=False, flush=True, verbose=0)
line,=ax.semilogy(freqs[4:]/1e6,abs(dd0[0,4:,0].T), label='ADC corr ROUND');

def animate(i):
    tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush_timeout=0.01, soft_integ_period=1, number_of_results=1, align=False, flush=False, verbose=0)
    line.set_ydata(abs(dd0[0,4:,0].T))
# animate(1)

# # ani = animation.FuncAnimation(fig, animate, interval=200, blit=True, save_count=50,repeat=True)
ani = animation.FuncAnimation(fig, animate, interval=3, repeat=True, cache_frame_data=False)
# while True:
#     tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush_timeout=0.01, soft_integ_period=1, number_of_results=1, align=False, flush=False, verbose=0)
#     line.set_ydata(abs(dd0[0,4:,0].T))
#     plt.show()

In [ ]:
ani

In [ ]:
plt.close()

In [ ]:
16384/3e9*16384*3

In [ ]:
fig, ax = plt.subplots()
ax.set_title(f'CORR output, Chan 0 autocorr')
# ax.set_xlabel('Frequency [Hz]')
ax.set_xlabel('Freq (MHz)')
ax.set_ylabel('Time (s)')
Finteg = i.CORR.INTEGRATION_PERIOD+1
dt=1/i.fpga.FRAME_RATE*Finteg
im = ax.imshow(10*np.log10(abs(dd0[:,:,0])), aspect='auto', interpolation='antialiased', extent=(0,Fs/2/1e6,0,dd0.shape[0]*dt))
fig.colorbar(im, ax=ax, label='colorbar')

In [ ]:
print(f'Freq resolution={Fs/16384} Hz, time resolution = {1/i.fpga.FRAME_RATE*16384} s')
i.CORR.INTEGRATION_PERIOD

## Validate the noise generator output

In [ ]:
i.set_channelizer(data_source='noise', offset_binary_encoding=False)
funcgen.SHIFT=0
# i.set_gains((4096,15))
#i.start_data_capture(0.03, source='scaler')

In [ ]:
### Check amplitude

In [ ]:
i.start_data_capture(0.1, source='adc')
t,(dt, *_), c = r.read_raw_frames(format='16')
# plt.plot(d[0])
# plt.title(f'FUNCGEN output, noise generator, shift={funcgen.SHIFT}');
fig, ax = plt.subplots()
ax.set_title(f'FUNCGEN output, noise generator, shift={funcgen.SHIFT}')
ax.set_xlabel('Sample #')
ax.set_ylabel('FUCGEN output')
ax.plot(dt)

In [ ]:
# Check FFT amplitude

In [ ]:
i.start_data_capture(0.1, source='scaler')
fft.FFT_SHIFT=(1<<10)-1
scaler.BYPASS = 1
scaler.DATA_TYPE = 0
_,(df, *_),_ = r.read_raw_frames(format='16+16', ncap=1, split=1)
fig, ax = plt.subplots()
ax.set_title(f'FFT output, noise generator, shift={funcgen.SHIFT}')
ax.set_xlabel('Sample #')
ax.set_ylabel('FFT output')
ax.plot(abs(df[0]))
d.shape

In [ ]:
### Check spectrum

In [ ]:
fig, ax = plt.subplots()
ax.set_label(f'FUNCGEN output, noise generator, shift={funcgen.SHIFT}')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Frequency [Hz]')
pxx,f,t,cax = ax.specgram(dt, Fs=3e9, mode='magnitude');
fig.colorbar(cax, label='dB');



In [ ]:
fig, ax = plt.subplots()
ax.set_title(f'FUNCGEN output, noise generator, shift={funcgen.SHIFT}')
ax.magnitude_spectrum(dt, Fs=3e9, scale='linear');
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Magnitude ');
# fig.colorbar(cax, label='dB');

In [ ]:
fig, ax = plt.subplots()
ax.set_title(f'FUNCGEN output, noise generator, shift={funcgen.SHIFT}')
ax.hist(dt, 256);
ax.set_xlabel('value')
ax.set_ylabel('count ');

In [ ]:
d[0].mean()

In [ ]:
d[0].std()

## Validate the scaler output

In [ ]:
i.set_gains(4096*2**15)
scaler.BYPASS=1
scaler.DATA_TYPE=1
ucap.MODE = 3  # single-channel mode, 16 frames/channel
i.start_data_capture(0.1, source='scaler')
ucap.OUTPUT_SOURCE_SEL=0
t,d,c = r.read_raw_frames(format='16+16')



In [ ]:
fig, ax = plt.subplots()
ax.set_title(f'SCALER output, 16+16 bit mode, flat gain, noise generator, shift={funcgen.SHIFT}')
ax.set_xlabel('Bin #')
ax.set_ylabel('Scaler output')
ax.plot(abs(d[0, :8192]));

In [ ]:
i.set_gains((np.linspace(0, 8191, 8192, dtype=np.int16), 15))

In [ ]:
i.set_gains((np.linspace(7000, 9000, 8192, dtype=np.int16), 15))

In [ ]:
# funcgen.SHIFT=7
i.start_data_capture(0.02, source='scaler')
ucap.OUTPUT_SOURCE_SEL=0
scaler.BYPASS=1
scaler.DATA_TYPE=1
ts,(k, *_), _ = r.read_raw_frames(format='16+16', ncap=50, split=True, verbose=0)

In [ ]:
# Capture SCALER output in 4+4 ROUND mode
i.start_data_capture(0.02, source='scaler')
ucap.MODE = 3  # single-channel mode, 16 frames/channel
ucap.OUTPUT_SOURCE_SEL=0
scaler.FOUR_BITS = 1
scaler.BYPASS=0
scaler.DATA_TYPE=0
scaler.ROUNDING_MODE = 1
ts, (k41, *_), _ = r.read_raw_frames(format='16+16', ncap=50, split=True, verbose=0)


In [ ]:
# Simulate 4-bit rounding on 16+16-bit data
H=4096
k41sim = ((k.real + H//2)//H*H) + 1j*((k.imag + H//2) //H*H)

In [ ]:
k41sim[0,8005]

In [ ]:
k[0,8005]

In [ ]:
abs(16*(1+1j)*(1<<12))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5,8))
ax.set_title(f'SCALER output, ramp gain, noise generator, shift={funcgen.SHIFT}')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('abs() of scaler output')
ax.plot(abs(k41[0]), 'm', label='4+4 Single frame');
ax.plot(abs(k41sim[0]), '0.5', label='4+4 SIM Single frame');
ax.plot(abs(k[0]), 'c', label='16+16 Single frame');

ax.plot(abs(k).mean(axis=0), 'b', label=f'16+16 Avg');
ax.plot(abs(k41).mean(axis=0), 'r', linewidth=2, label=f'4+4 Avg');
ax.plot(abs(k41sim).mean(axis=0), 'k', linewidth=0.8, label=f'4+4 SIM Avg');
# ax.hlines(abs(8192*(1+1j)), 0, 8191)
ax.legend();

In [ ]:
fig, ax = plt.subplots()
ax.plot(abs(k[0]))
ax.plot(abs(k41sim[0]))


In [ ]:
fig, ax = plt.subplots()
ax.set_title(f'SCALER output, 4+4 bit ROUND mode, ramp gain, noise generator, shift={funcgen.SHIFT}')
ax.set_xlabel('bin #')
ax.set_ylabel('abs() of scaler output')
ax.plot(abs(k41[0]), label='Single frame');
ax.plot(abs(k41).mean(axis=0), 'r', label=f'Average of {len(k41)} frames');
ax.legend()

## Validate scaler output
We capture, autocorrelate and integrate the 4+4 scaler output, then compare with the 16+16 output scaled by software 


In [ ]:
def autocorr(v):
    re = v.real.astype('i8') 
    im = v.imag.astype('i8') 
    return (re**2 + im**2)

def autocorr4(v):
    re = (v.real.astype('i8') + (1<<11)) >> 12 
    im = (v.imag.astype('i8') + (1<<11)) >> 12 
    return (re**2 + im**2)


In [ ]:
ucap.OUTPUT_SOURCE_SEL = 1
ucorr.NO_ACCUM = 0
scaler.BYPASS=1
scaler.DATA_TYPE=1
tt,dd,cc,ss = cr.read_corr_frames(flush_timeout=0.01)

In [ ]:
ucap.OUTPUT_SOURCE_SEL = 1
ucorr.NO_ACCUM = 0
scaler.FOUR_BITS = 1
scaler.SATURATE_ON_MINUS_7 = 1
scaler.BYPASS=0
scaler.DATA_TYPE=0
scaler.ROUNDING_MODE = 0
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush_timeout=0.01, verbose=0)

In [ ]:
ucap.OUTPUT_SOURCE_SEL = 1
ucorr.NO_ACCUM = 0
scaler.FOUR_BITS = 1
scaler.BYPASS=0
scaler.DATA_TYPE=0
scaler.ROUNDING_MODE = 1
_,dd1,_,_ = cr.read_corr_frames(flush_timeout=0.01, verbose=0)

In [ ]:
ucap.OUTPUT_SOURCE_SEL = 1
ucorr.NO_ACCUM = 0
scaler.FOUR_BITS = 1
scaler.BYPASS=0
scaler.DATA_TYPE=0
scaler.ROUNDING_MODE = 2
_,dd2,_,_ = cr.read_corr_frames(flush_timeout=0.01, verbose=0)

In [ ]:
fig, ax = plt.subplots(figsize=(9.5,5), layout='tight')
ax.set_title(f'CORR output, Chan 0 autocorr\nramp gain, noise generator, shift={funcgen.SHIFT}')
# ax.set_xlabel('Frequency [Hz]')
ax.set_xlabel('Bin #')
ax.set_ylabel('CORR output')
ax.plot(abs(dd[0,4:,0]), linewidth=2, label='16+16')
ax.plot(abs(dd0[0,4:,0]), 'k', linewidth=0.5, label='4+4 TRUNC')
ax.plot(abs(dd1[0,4:,0]), 'r', linewidth=2, label='4+4 ROUND')
ax.plot(abs(dd2[0,4:,0]), 'g', linewidth=1, label='4+4 CONV ROUND')
ax.plot(autocorr(k).mean(axis=0)/(2**12)*4, 'b', label="16+16 soft corr", linewidth=0.5);
ax.plot(autocorr4(k).mean(axis=0)*32768/2, 'y', label="4+4 ROUND soft corr");
ax.legend(shadow=True, title="Scaler mode");

In [ ]:
scaler.FOUR_BITS

In [ ]:
kr.dtype

In [ ]:
kr.shape

In [ ]:
kr[0,8000:8100]

In [ ]:
kk[8000]

In [ ]:
(kr**2).dtype

In [ ]:
k=np.random.randint(-50, 50, (5000,8192)) * np.linspace(0, 8191, 8192)[None,:]
H=4096
k41 = ((k+H//2)//H)*H
hex(int(k[0,8000])), hex(int(k41[0,8000]))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5,8))
ax.set_title(f'Simulated random signal, ramp gain')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('abs() of scaler output')
# ax.plot(abs(k41sim[0]), '0.5', label='4+4 SIM Single frame');
ax.plot(abs(k[0]), 'c', label='16+16 Single frame');
ax.plot(abs(k41[0]), 'm', label='4+4 Single frame');

ax.plot(abs(k).mean(axis=0), 'b', label=f'16+16 Avg');
ax.plot(abs(k41).mean(axis=0), 'r', linewidth=2, label=f'4+4 Avg');
# ax.plot(k41sim.mean(axis=0), 'k', linewidth=0.8, label=f'4+4 SIM Avg');
# ax.hlines(abs(8192*(1+1j)), 0, 8191)
ax.legend();

In [ ]:
bin(fft.FFT_SHIFT)

In [ ]:
A = 1
N = 50000
# k = np.random.randint(-A, A, (N,))
k = np.random.normal(0, A, size=(N,)).astype('i4')
# H = 4096
# k41 = ((k+H//2)//H)*H
# hex(int(k[0,8000])), hex(int(k41[0,8000]))
print(f'FFT mean={k.mean()}, std={k.std()}, min={k.min()}, max={k.max()}')

In [ ]:
149/86

In [ ]:
bin_edges=np.linspace(k.min()-0.5,k.max()+0.5,k.max()-k.min()+2)
bin_centers = np.linspace(k.min(),k.max(),k.max()-k.min()+1)

In [ ]:
h,v=np.histogram(k, bin_edges)
h = h/len(k)
h.shape

In [ ]:
def gauss(x, var):
    return np.exp(-(x/var)**2 / 2)
v=bin_centers=np.linspace(-4*A, 4*A, 8*A+1)
h = gauss(v,A)
h = h/sum(h)
bin_centers
v.shape,h.shape
print(f'mean={(h*v).sum()}')
print(f'std={np.sqrt((h*v**2).sum())}')

In [ ]:
print(f'avg(k)={k.mean()}')
print(f'h*v={(h*bin_centers).sum()}')

In [ ]:
fig, ax = plt.subplots(figsize=(9.5,8))
ax.set_title(f'FFT output')
ax.set_xlabel('time')
ax.set_ylabel('FFT output for one bin')
ax.plot(k);

In [ ]:
fig, ax = plt.subplots(figsize=(9.5,8))
ax.set_title(f'Histogram of FFT output')
ax.set_xlabel('value')
ax.set_ylabel('count')
ax.bar(bin_centers, h);
# ax.hist(k,400)

In [ ]:
H=(1<<(35-4))
ideal_gain = 2.4/k.std()*H
#glog = 17
glog = np.log2(ideal_gain)//1-11
glin = np.linspace(0, 4096-1, 4096)
g = glin*(2**glog)/H
hh = bin_centers[None,:]*g[:,None]
ideal_glin = 2.4/k.std()*H/2**glog
print(f'computed glog = {glog}, ideal_glin={ideal_glin}')
ideal_glin

In [ ]:
fig, ax = plt.subplots(figsize=(9.5,8))
ax.set_title(f'Scaler output')
ax.set_xlabel('time')
ax.set_ylabel('FFT output for one bin')
ax.plot(k//H, label='Original FFT value');
ax.plot((k*ideal_gain)//H, label='4-bit Quantized FFT value');

In [ ]:
h41 = ((hh +.5)//1).clip(-7, 7)
h41 = (np.round(hh)//1).clip(-7, 7)
h41 = (np.round(hh)//1)

In [ ]:
np.round(1.5)

In [ ]:
z=((hh%1 == 0.5)*h).sum(axis=1)
fig, ax = plt.subplots()
ax.plot(z)

In [ ]:
def op(x):
    return abs(x)**2

In [ ]:
# hh41=(op(h41)*h).sum(axis=1)
# hh00=(op(hh)*h).sum(axis=1)
hh41=np.sqrt((op(h41)*h).sum(axis=1))
hh00=np.sqrt((op(hh)*h).sum(axis=1))
hh41.shape

In [ ]:
1/2.2*8

In [ ]:
hh00[-1]

In [ ]:
fig, ax = plt.subplots(figsize=(9.5,8))
ax.set_title(f'Effective vs real expected value as a function of gain\n 18+18 bit pre-scaler FFT={k.std():.1f} lsb RMS, glog={glog}')
ax.set_xlabel('Gain (glin)')
ax.set_ylabel('Post-gain value (lsb RMS)')
ax.plot(glin, hh41, label= 'real value');
ax.plot(glin, hh00, label= 'expected value');
ax.vlines(ideal_glin, *ax.get_ylim(),'r', label='gain for std value of 2.4')
ax.legend()


In [ ]:
ax.get_ylim()

In [ ]:
2.4**2

In [ ]:
375*3



In [ ]:
i.get_temperatures()

In [ ]:
1100/1500*8192

In [ ]:
16384*5e-6


In [ ]:
77152-60768

In [ ]:
200e6/16/12

In [ ]:
1040+3000

In [ ]:
2.4-1.5

In [ ]:
sorted([1040+n*3000 for n in range(5)] + [-1040+n*3000 for n in range(1,5)])

In [ ]:
1.5+(1.5-1.04)

In [ ]:
def get_casper_fft_descramble(n_bit_fft, n_bit_parallel):
    """
    Get the descramble map for a CASPER FFT with
    2**n_bit_fft channels, presenting 2**n_bit_parallel
    on each cycle
    """
    n_fft = 2**n_bit_fft
    n_parallel = 2**n_bit_parallel
    return np.arange(n_fft).reshape(n_fft // n_parallel, n_parallel).transpose().flatten()

In [ ]:
def fft_descramble_map(n_bit_fft, n_bit_parallel):
    n_fft = 2**n_bit_fft
    n_parallel = 2**n_bit_parallel
    f_nat = np.arange(n_fft)
    return ((f_nat >> (n_bit_fft - n_bit_parallel)) | (f_nat << n_bit_parallel)) & (n_fft -1)

In [ ]:
m1=get_casper_fft_descramble(13,2)
m2=fft_descramble_map(13,2)
all(m1==m2)

In [ ]:
timeit m1=get_casper_fft_descramble(13,2)

In [ ]:
timeit m2=fft_descramble_map(13,2)

In [ ]:
for b in range(13):
    F=2**b
    FF = m[F] 
    print(f'{F:013b} -> {FF:013b}')

In [ ]:
a=np.arange(8192, dtype=np.int16)
b=a.reshape(8192//4,4)

In [ ]:
b

In [ ]:
b.T

In [ ]:
b.T.flatten()

In [ ]:
ca = await FPGAArray('crs 0032', stdout_log_level='info').init()

In [ ]:
ca.is_fpga_programmed()

In [ ]:
i=ca.ib[0]

In [ ]:
await i.is_fpga_programmed_async()